# Task 6: Bidirectional Long Short-Term Memory (BiLSTM) Cell Mechanics from Scratch

**Objective:** Build recurrent gate architectures, memory cell updates, and bidirectional sequence routing from scratch using only raw PyTorch tensor operations.

### Gated Recurrent Equations
For each recurrent step $t$:
1. **Forget Gate:** $f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)$
2. **Input Gate:** $i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)$
3. **Candidate Cell State:** $\tilde{C}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$
4. **Cell State Update:** $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
5. **Output Gate:** $o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)$
6. **Hidden State Update:** $h_t = o_t \odot \tanh(C_t)$

In a Bidirectional LSTM, we run the sequence forward ($0 \dots T-1$) and backward ($T-1 \dots 0$), concatenating the final hidden outputs $[h_t^{forward}; h_t^{backward}]$.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class LSTMCellScratch:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # Combine weights for all 4 gates (Forget, Input, Candidate, Output) to optimize dot products
        # Shape: (input_dim + hidden_dim, 4 * hidden_dim)
        self.W = torch.randn(input_dim + hidden_dim, 4 * hidden_dim) * np.sqrt(2.0 / (input_dim + hidden_dim))
        self.b = torch.zeros(4 * hidden_dim)
        
    def forward(self, x, h_prev, c_prev):
        # x: (batch_size, input_dim)
        # h_prev: (batch_size, hidden_dim)
        # c_prev: (batch_size, hidden_dim)
        
        # Concatenate inputs
        combined = torch.cat([x, h_prev], dim=-1)
        
        # Calculate linear projection for all gates
        gates = torch.matmul(combined, self.W) + self.b
        
        # Split projections into 4 gates
        f_gate, i_gate, c_cand, o_gate = torch.chunk(gates, 4, dim=-1)
        
        # Non-linear activations
        f = torch.sigmoid(f_gate)
        i = torch.sigmoid(i_gate)
        c_tilde = torch.tanh(c_cand)
        o = torch.sigmoid(o_gate)
        
        # State updates
        c = f * c_prev + i * c_tilde
        h = o * torch.tanh(c)
        
        return h, c

class BidirectionalLSTMScratch:
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        # Initialize forward and backward cells
        self.forward_cell = LSTMCellScratch(input_dim, hidden_dim)
        self.backward_cell = LSTMCellScratch(input_dim, hidden_dim)
        
    def forward(self, X, seq_lengths):
        # X: (batch_size, seq_len, input_dim)
        # seq_lengths: (batch_size,) - list of actual lengths for each sequence in the batch
        batch_size, seq_len, input_dim = X.shape
        
        # Initial hidden and cell states
        h_f = torch.zeros(batch_size, self.hidden_dim)
        c_f = torch.zeros(batch_size, self.hidden_dim)
        h_b = torch.zeros(batch_size, self.hidden_dim)
        c_b = torch.zeros(batch_size, self.hidden_dim)
        
        forward_outputs = []
        backward_outputs = [None] * seq_len
        
        # 1. Forward Pass
        for t in range(seq_len):
            x_t = X[:, t, :]
            h_f, c_f = self.forward_cell.forward(x_t, h_f, c_f)
            
            # Handle padding: if step exceeds sequence length, keep previous cell state
            mask = (t < seq_lengths).float().unsqueeze(-1)
            h_f = mask * h_f + (1 - mask) * (forward_outputs[-1] if forward_outputs else 0)
            
            forward_outputs.append(h_f)
            
        # 2. Backward Pass
        for t in reversed(range(seq_len)):
            x_t = X[:, t, :]
            h_b, c_b = self.backward_cell.forward(x_t, h_b, c_b)
            
            # Handle padding: mask backward inputs if index exceeds sequence length
            mask = (t < seq_lengths).float().unsqueeze(-1)
            h_b = mask * h_b
            
            backward_outputs[t] = h_b
            
        # Stack outputs
        f_out = torch.stack(forward_outputs, dim=1)
        b_out = torch.stack(backward_outputs, dim=1)
        
        # Concatenate forward and backward representation
        out = torch.cat([f_out, b_out], dim=-1)
        return out

In [ ]:
# Setup simple inputs and verify shape routing
batch_size = 4
seq_len = 10
input_dim = 16
hidden_dim = 32

X = torch.randn(batch_size, seq_len, input_dim)
seq_lengths = torch.tensor([10, 8, 6, 9]) # Varying input sequence lengths in the batch

custom_bilstm = BidirectionalLSTMScratch(input_dim, hidden_dim)
outputs = custom_bilstm.forward(X, seq_lengths)

print("Input Tensor Shape: ", X.shape)
print("Output Tensor Shape:", outputs.shape)
assert outputs.shape == (batch_size, seq_len, 2 * hidden_dim), "Verification failed: incorrect shape output!"
print("Success: Bidirectional LSTM correctly handles masking and forward/backward hidden states stacking!")